# Finland Garrison Health Data - HANA Persistence

This notebook loads Finnish conscript health data into SAP HANA database for RAG-based epidemic monitoring.

## Step 1: Data Exploration

Analyze all 5 Excel files to understand their structure

In [1]:
import pandas as pd
import os
from datetime import datetime

# List all Excel files
excel_files = [
    'FDF conscript health data.xlsx',
    'Hyvinvointialueiden kunnat.xlsx',
    'THL_Vuosi_2025.xlsx',
    'THL_Vuosi_2026.xlsx',
    'varusmiehet_organisaatio.xlsx'
]

print("Excel files in data folder:")
for f in excel_files:
    path = f'data/{f}'
    if os.path.exists(path):
        print(f"✓ {f}")
    else:
        print(f"✗ {f} NOT FOUND")

Excel files in data folder:
✓ FDF conscript health data.xlsx
✓ Hyvinvointialueiden kunnat.xlsx
✓ THL_Vuosi_2025.xlsx
✓ THL_Vuosi_2026.xlsx
✓ varusmiehet_organisaatio.xlsx


### 1. FDF Conscript Health Data

In [2]:
# Load FDF conscript health data
df_fdf_health = pd.read_excel('data/FDF conscript health data.xlsx')

print(f"Shape: {df_fdf_health.shape}")
print(f"\nColumns: {list(df_fdf_health.columns)}")
print(f"\nData types:\n{df_fdf_health.dtypes}")
print(f"\nFirst 5 rows:")
df_fdf_health.head()

Shape: (3000, 3)

Columns: ['ID', 'Taudin nimi', 'Tautiluokitus']

Data types:
ID                int64
Taudin nimi      object
Tautiluokitus    object
dtype: object

First 5 rows:


,ID,Taudin nimi,Tautiluokitus
0,1,Rinovirus,J00
1,2,Rinovirus,J00
2,3,Rinovirus,J00
3,4,Influenssa,J10
4,5,Rinovirus,J00


In [3]:
# Check unique diseases
print("Unique diseases:")
print(df_fdf_health['Taudin nimi'].value_counts())
print(f"\nUnique disease codes:")
print(df_fdf_health['Tautiluokitus'].value_counts())

Unique diseases:
Taudin nimi
Rinovirus                 1212
Influenssa                 722
Koronavirus SARS-CoV-2     635
Norovirus                  275
Adenovirus                 156
Name: count, dtype: int64

Unique disease codes:
Tautiluokitus
J00      1212
J10       722
U07.1     635
A08.1     275
B97.0     156
Name: count, dtype: int64


### 2. Municipality to Wellbeing Services County Mapping

In [4]:
# Load municipality mapping
df_municipality = pd.read_excel('data/Hyvinvointialueiden kunnat.xlsx')

print(f"Shape: {df_municipality.shape}")
print(f"\nColumns: {list(df_municipality.columns)}")
print(f"\nData types:\n{df_municipality.dtypes}")
print(f"\nFirst 5 rows:")
df_municipality.head()

Shape: (30, 22)

Columns: ['Etelä-Karjalan hyvinvointialue', 'Etelä-Pohjanmaan hyvinvointialue', 'Etelä-Savon hyvinvointialue', 'Itä-Uusimaan hyvinvointialue', 'Kainuun hyvinvointialue', 'Kanta-Hämeen hyvinvointialue', 'Keski-Pohjanmaan hyvinvointialue', 'Keski-Suomen hyvinvointialue', 'Keski-Uudenmaan hyvinvointialue', 'Kymenlaakson hyvinvointialue', 'Lapin hyvinvointialue', 'Länsi-Uudenmaan hyvinvointialue', 'Pirkanmaan hyvinvointialue', 'Pohjanmaan hyvinvointialue', 'Pohjois-Karjalan hyvinvointialue', 'Pohjois-Pohjanmaan hyvinvointialue', 'Pohjois-Savon hyvinvointialue', 'Päijät-Hämeen hyvinvointialue', 'Satakunnan hyvinvointialue', 'Vantaan ja Keravan hyvinvointialue', 'Varsinais-Suomen hyvinvointialue', 'Helsingin kaupunki']

Data types:
Etelä-Karjalan hyvinvointialue        object
Etelä-Pohjanmaan hyvinvointialue      object
Etelä-Savon hyvinvointialue           object
Itä-Uusimaan hyvinvointialue          object
Kainuun hyvinvointialue               object
Kanta-Hämeen hyvinvoin

,Etelä-Karjalan hyvinvointialue,Etelä-Pohjanmaan hyvinvointialue,Etelä-Savon hyvinvointialue,Itä-Uusimaan hyvinvointialue,Kainuun hyvinvointialue,Kanta-Hämeen hyvinvointialue,Keski-Pohjanmaan hyvinvointialue,Keski-Suomen hyvinvointialue,Keski-Uudenmaan hyvinvointialue,Kymenlaakson hyvinvointialue,...,Pirkanmaan hyvinvointialue,Pohjanmaan hyvinvointialue,Pohjois-Karjalan hyvinvointialue,Pohjois-Pohjanmaan hyvinvointialue,Pohjois-Savon hyvinvointialue,Päijät-Hämeen hyvinvointialue,Satakunnan hyvinvointialue,Vantaan ja Keravan hyvinvointialue,Varsinais-Suomen hyvinvointialue,Helsingin kaupunki
0,Imatra,Alajärvi,Enonkoski,Askola,Hyrynsalmi,Forssa,Halsua,Hankasalmi,Hyvinkää,Hamina,...,Akaa,Kaskinen,Heinävesi,Alavieska,Iisalmi,Asikkala,Eura,Vantaa,Aura,Helsinki
1,Lappeenranta,Alavus,Hirvensalmi,Lapinjärvi,Kajaani,Hattula,Kannus,Joutsa,Järvenpää,Kotka,...,Hämeenkyrö,Korsnäs,Ilomantsi,Haapajärvi,Joroinen,Hartola,Eurajoki,Kerava,Kaarina,NaN
2,Lemi,Evijärvi,Juva,Loviisa,Kuhmo,Hausjärvi,Kaustinen,Jyväskylä,Nurmijärvi,Kouvola,...,Ikaalinen,Kristiinankaupunki,Joensuu,Haapavesi,Kaavi,Heinola,Harjavalta,NaN,Kemiönsaari,NaN
3,Luumäki,Ilmajoki,Kangasniemi,Myrskylä,Paltamo,Humppila,Kokkola,Jämsä,Mäntsälä,Miehikkälä,...,Juupajoki,Kruunupyy,Juuka,Hailuoto,Keitele,Hollola,Huittinen,NaN,Koski Tl,NaN
4,Parikkala,Isojoki,Mikkeli,Porvoo,Puolanka,Hämeenlinna,Lestijärvi,Kannonkoski,Tuusula,Pyhtää,...,Kangasala,Laihia,Kitee,Ii,Kiuruvesi,Iitti,Jämijärvi,NaN,Kustavi,NaN


### 3. Conscript Organization Data

In [5]:
# Load conscript organization data
df_organization = pd.read_excel('data/varusmiehet_organisaatio.xlsx')

print(f"Shape: {df_organization.shape}")
print(f"\nColumns: {list(df_organization.columns)}")
print(f"\nData types:\n{df_organization.dtypes}")
print(f"\nFirst 5 rows:")
df_organization.head()

Shape: (3000, 5)

Columns: ['ID', 'Joukko-osasto', 'Joukkoyksikkö', 'Kotikunta', 'Saapumiserä']

Data types:
ID                int64
Joukko-osasto    object
Joukkoyksikkö    object
Kotikunta        object
Saapumiserä      object
dtype: object

First 5 rows:


,ID,Joukko-osasto,Joukkoyksikkö,Kotikunta,Saapumiserä
0,1,Kainuun prikaati,Esikuntakomppania,Oulu,1/26
1,2,Kainuun prikaati,Esikuntakomppania,Raahe,2/25
2,3,Kainuun prikaati,Tykistöpatteri,Nurmes,1/26
3,4,Rannikkoprikaati,Huoltokomppania,Vaasa,1/25
4,5,Rannikkoprikaati,Rannikkopataljoona,Espoo,1/26


### 4. THL Disease Data 2025

In [6]:
# Load THL 2025 data
df_thl_2025 = pd.read_excel('data/THL_Vuosi_2025.xlsx')

print(f"Shape: {df_thl_2025.shape}")
print(f"\nColumns: {list(df_thl_2025.columns)}")
print(f"\nData types:\n{df_thl_2025.dtypes}")
print(f"\nFirst 5 rows:")
df_thl_2025.head()

Shape: (37, 1273)

Columns: ['Tapaukset', 'Vuosi 2025 Viikko 01', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Vuosi 2025 Viikko 02', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Vuosi 2025 Viikko 03', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', 'Unnamed: 57', 'Unnamed: 58', 'Unnamed: 59', 'Unnamed: 60', 'Unnamed: 61', 'Unnamed: 62', 'Unnamed:

/Users/I745534/Library/CloudStorage/OneDrive-SAPSE/Documents/Daedalus Alpha/DEIG/Hackathon/3 Oslo/ai_hackathon_oslo/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Tapaukset,Vuosi 2025 Viikko 01,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 1263,Unnamed: 1264,Unnamed: 1265,Unnamed: 1266,Unnamed: 1267,Unnamed: 1268,Unnamed: 1269,Unnamed: 1270,Unnamed: 1271,Unnamed: 1272
0,Tietopoiminta,Itä-Uudenmaan hyvinvointialue,Keski-Uudenmaan hyvinvointialue,Länsi-Uudenmaan hyvinvointialue,Vantaan ja Keravan hyvinvointialue,Varsinais-Suomen hyvinvointialue,Satakunnan hyvinvointialue,Kanta-Hämeen hyvinvointialue,Pirkanmaan hyvinvointialue,Päijät-Hämeen hyvinvointialue,...,Keski-Suomen hyvinvointialue,Etelä-Pohjanmaan hyvinvointialue,Pohjanmaan hyvinvointialue,Keski-Pohjanmaan hyvinvointialue,Pohjois-Pohjanmaan hyvinvointialue,Kainuun hyvinvointialue,Lapin hyvinvointialue,Helsingin kaupunki,Ahvenanmaa,Kaikki hyvinvointialueet
1,Influenssa,5,10,18,13,36,16,9,35,12,...,949,1106,499,333,2244,386,452,3113,125,25402
2,Influenssa A,5,9,16,8,33,15,8,33,9,...,835,945,403,297,1979,342,344,2718,115,22373
3,Influenssa B,0,1,2,5,3,1,1,2,3,...,114,161,96,36,265,44,108,391,10,2943
4,Adenovirus,0,2,2,0,10,0,0,2,0,...,75,134,122,38,448,57,118,322,9,3011


### 5. THL Disease Data 2026

In [7]:
# Load THL 2026 data
df_thl_2026 = pd.read_excel('data/THL_Vuosi_2026.xlsx')

print(f"Shape: {df_thl_2026.shape}")
print(f"\nColumns: {list(df_thl_2026.columns)[:10]}")  # First 10 columns
print(f"\nData types (first 10):\n{df_thl_2026.dtypes[:10]}")
print(f"\nFirst 5 rows:")
df_thl_2026.head()

Shape: (37, 145)

Columns: ['Tapaukset', 'Vuosi 2026 Viikko 01', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9']

Data types (first 10):
Tapaukset               object
Vuosi 2026 Viikko 01    object
Unnamed: 2              object
Unnamed: 3              object
Unnamed: 4              object
Unnamed: 5              object
Unnamed: 6              object
Unnamed: 7              object
Unnamed: 8              object
Unnamed: 9              object
dtype: object

First 5 rows:


/Users/I745534/Library/CloudStorage/OneDrive-SAPSE/Documents/Daedalus Alpha/DEIG/Hackathon/3 Oslo/ai_hackathon_oslo/.venv/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Tapaukset,Vuosi 2026 Viikko 01,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 135,Unnamed: 136,Unnamed: 137,Unnamed: 138,Unnamed: 139,Unnamed: 140,Unnamed: 141,Unnamed: 142,Unnamed: 143,Unnamed: 144
0,Tietopoiminta,Itä-Uudenmaan hyvinvointialue,Keski-Uudenmaan hyvinvointialue,Länsi-Uudenmaan hyvinvointialue,Vantaan ja Keravan hyvinvointialue,Varsinais-Suomen hyvinvointialue,Satakunnan hyvinvointialue,Kanta-Hämeen hyvinvointialue,Pirkanmaan hyvinvointialue,Päijät-Hämeen hyvinvointialue,...,Keski-Suomen hyvinvointialue,Etelä-Pohjanmaan hyvinvointialue,Pohjanmaan hyvinvointialue,Keski-Pohjanmaan hyvinvointialue,Pohjois-Pohjanmaan hyvinvointialue,Kainuun hyvinvointialue,Lapin hyvinvointialue,Helsingin kaupunki,Ahvenanmaa,Kaikki hyvinvointialueet
1,Influenssa,23,51,97,59,87,30,43,98,27,...,84,105,85,73,301,16,75,770,30,4274
2,Influenssa A,23,51,93,56,84,30,42,95,26,...,83,104,85,73,298,16,75,760,30,4228
3,Influenssa B,0,0,4,3,1,0,1,3,1,...,1,1,0,0,3,0,0,10,0,43
4,Adenovirus,0,3,0,1,3,4,0,3,2,...,2,2,3,1,14,5,5,16,0,124


## Step 2: Data Transformation

Transform THL data from wide format to normalized format for HANA

In [8]:
# Transform THL data from wide to long format
def transform_thl_data(df, year):
    """
    Transform THL data from wide format to normalized long format
    Each row will be: disease_name, hyvinvointialue, week, year, case_count
    """
    # First row contains hyvinvointialue names
    hyvinvointialue_names = df.iloc[0, 1:].values
    
    # Get disease names from first column (skip first row which is header)
    disease_names = df.iloc[1:, 0].values
    
    # Get the case counts (skip first row and first column)
    data_rows = []
    
    for disease_idx, disease_name in enumerate(disease_names):
        # Get all values for this disease (skip first column which is disease name)
        disease_row = df.iloc[disease_idx + 1, 1:]
        
        # Each value corresponds to a hyvinvointialue
        for hva_idx, case_count in enumerate(disease_row):
            if hva_idx < len(hyvinvointialue_names):
                hva_name = hyvinvointialue_names[hva_idx]
                # Parse week from column name if available
                week = 1  # Default, will be refined
                
                data_rows.append({
                    'disease_name': disease_name,
                    'hyvinvointialue': hva_name,
                    'year': year,
                    'week': week,
                    'case_count': case_count if pd.notna(case_count) else 0
                })
    
    return pd.DataFrame(data_rows)

# Transform both years
print("Transforming THL 2025 data...")
df_thl_2025_transformed = transform_thl_data(df_thl_2025, 2025)
print(f"Transformed shape: {df_thl_2025_transformed.shape}")
print(df_thl_2025_transformed.head(10))

print("\nTransforming THL 2026 data...")
df_thl_2026_transformed = transform_thl_data(df_thl_2026, 2026)
print(f"Transformed shape: {df_thl_2026_transformed.shape}")
print(df_thl_2026_transformed.head(10))

Transforming THL 2025 data...
Transformed shape: (45792, 5)
  disease_name                     hyvinvointialue  year  week case_count
0   Influenssa       Itä-Uudenmaan hyvinvointialue  2025     1          5
1   Influenssa     Keski-Uudenmaan hyvinvointialue  2025     1         10
2   Influenssa     Länsi-Uudenmaan hyvinvointialue  2025     1         18
3   Influenssa  Vantaan ja Keravan hyvinvointialue  2025     1         13
4   Influenssa    Varsinais-Suomen hyvinvointialue  2025     1         36
5   Influenssa          Satakunnan hyvinvointialue  2025     1         16
6   Influenssa        Kanta-Hämeen hyvinvointialue  2025     1          9
7   Influenssa          Pirkanmaan hyvinvointialue  2025     1         35
8   Influenssa       Päijät-Hämeen hyvinvointialue  2025     1         12
9   Influenssa        Kymenlaakson hyvinvointialue  2025     1          5

Transforming THL 2026 data...
Transformed shape: (5184, 5)
  disease_name                     hyvinvointialue  year  week cas

## Step 3: Normalize Municipality Mapping Data

In [9]:
# Transform municipality mapping from wide to long format
municipality_rows = []

for col in df_municipality.columns:
    hyvinvointialue = col
    municipalities = df_municipality[col].dropna().values
    
    for municipality in municipalities:
        municipality_rows.append({
            'hyvinvointialue': hyvinvointialue,
            'municipality': municipality
        })

df_municipality_normalized = pd.DataFrame(municipality_rows)
print(f"Normalized municipality mapping shape: {df_municipality_normalized.shape}")
print(df_municipality_normalized.head(20))

Normalized municipality mapping shape: (293, 2)
                     hyvinvointialue  municipality
0     Etelä-Karjalan hyvinvointialue        Imatra
1     Etelä-Karjalan hyvinvointialue  Lappeenranta
2     Etelä-Karjalan hyvinvointialue          Lemi
3     Etelä-Karjalan hyvinvointialue       Luumäki
4     Etelä-Karjalan hyvinvointialue     Parikkala
5     Etelä-Karjalan hyvinvointialue     Rautjärvi
6     Etelä-Karjalan hyvinvointialue    Ruokolahti
7     Etelä-Karjalan hyvinvointialue   Savitaipale
8     Etelä-Karjalan hyvinvointialue   Taipalsaari
9   Etelä-Pohjanmaan hyvinvointialue      Alajärvi
10  Etelä-Pohjanmaan hyvinvointialue        Alavus
11  Etelä-Pohjanmaan hyvinvointialue      Evijärvi
12  Etelä-Pohjanmaan hyvinvointialue      Ilmajoki
13  Etelä-Pohjanmaan hyvinvointialue       Isojoki
14  Etelä-Pohjanmaan hyvinvointialue       Isokyrö
15  Etelä-Pohjanmaan hyvinvointialue      Karijoki
16  Etelä-Pohjanmaan hyvinvointialue     Kauhajoki
17  Etelä-Pohjanmaan hyvinvointial

## Step 4: HANA Connection and Schema Setup

In [11]:
import os
from dotenv import load_dotenv
from hdbcli import dbapi

# Load environment variables
load_dotenv()

HANA_HOST = os.getenv("HANA_HOST")
HANA_PORT = os.getenv("HANA_PORT")
HANA_USER = os.getenv("HANA_HACKATHON_USER")
HANA_PASSWORD = os.getenv("HANA_HACKATHON_PASSWORD")

# Connect to HANA
conn = dbapi.connect(
    address=HANA_HOST,
    port=HANA_PORT,
    user="HACKATHON_USR",
    password=HANA_PASSWORD
)

cursor = conn.cursor()
print("Connected to HANA successfully!")

# Use existing schema
SCHEMA = "HACKATHON_USR"
print(f"Using schema: {SCHEMA}")

Connected to HANA successfully!
Using schema: HACKATHON_USR


## Step 5: Create HANA Tables

Create normalized tables for Finnish garrison health data

In [21]:
# Drop tables if they exist
tables_to_drop = [
    'FI_CONSCRIPT_HEALTH',
    'FI_MUNICIPALITY_MAPPING',
    'FI_CONSCRIPT_ORGANIZATION',
    'FI_THL_DISEASE_CASES',
    'FI_DISEASE_INFO',
    'FI_GARRISON_INFO'
]

for table in tables_to_drop:
    try:
        cursor.execute(f'DROP TABLE "{SCHEMA}"."{table}"')
        print(f"Dropped table {table}")
    except Exception as e:
        print(f"Table {table} does not exist or could not be dropped: {e}")

Table FI_CONSCRIPT_HEALTH does not exist or could not be dropped: (259, 'invalid table name: FI_CONSCRIPT_HEALTH: line 1 col 28 (at pos 27)')
Table FI_MUNICIPALITY_MAPPING does not exist or could not be dropped: (259, 'invalid table name: FI_MUNICIPALITY_MAPPING: line 1 col 28 (at pos 27)')
Table FI_CONSCRIPT_ORGANIZATION does not exist or could not be dropped: (259, 'invalid table name: FI_CONSCRIPT_ORGANIZATION: line 1 col 28 (at pos 27)')
Table FI_THL_DISEASE_CASES does not exist or could not be dropped: (259, 'invalid table name: FI_THL_DISEASE_CASES: line 1 col 28 (at pos 27)')
Table FI_DISEASE_INFO does not exist or could not be dropped: (259, 'invalid table name: FI_DISEASE_INFO: line 1 col 28 (at pos 27)')
Table FI_GARRISON_INFO does not exist or could not be dropped: (259, 'invalid table name: FI_GARRISON_INFO: line 1 col 28 (at pos 27)')


In [22]:
# Create Disease Information Master Table
create_disease_info_sql = f'''
CREATE COLUMN TABLE "{SCHEMA}"."FI_DISEASE_INFO" (
    "DISEASE_CODE" NVARCHAR(20) PRIMARY KEY,
    "DISEASE_NAME_FI" NVARCHAR(200),
    "DISEASE_NAME_EN" NVARCHAR(200),
    "SYMPTOMS" NVARCHAR(2000),
    "PREVENTION" NVARCHAR(2000),
    "AVERAGE_DURATION_DAYS" INTEGER,
    "HOW_IT_SPREADS" NVARCHAR(1000),
    "EPIDEMIC_THRESHOLD" INTEGER
)
'''

try:
    cursor.execute(create_disease_info_sql)
    print("✓ FI_DISEASE_INFO table created successfully!")
except Exception as e:
    print(f"FI_DISEASE_INFO table: {e}")

✓ FI_DISEASE_INFO table created successfully!


In [23]:
# Create Garrison Information Table
create_garrison_info_sql = f'''
CREATE COLUMN TABLE "{SCHEMA}"."FI_GARRISON_INFO" (
    "GARRISON_NAME" NVARCHAR(200) PRIMARY KEY,
    "LOCATION" NVARCHAR(200),
    "CAPACITY" INTEGER,
    "DESCRIPTION" NVARCHAR(1000)
)
'''

try:
    cursor.execute(create_garrison_info_sql)
    print("✓ FI_GARRISON_INFO table created successfully!")
except Exception as e:
    print(f"FI_GARRISON_INFO table: {e}")

✓ FI_GARRISON_INFO table created successfully!


In [24]:
# Create Municipality to Hyvinvointialue Mapping Table
create_municipality_sql = f'''
CREATE COLUMN TABLE "{SCHEMA}"."FI_MUNICIPALITY_MAPPING" (
    "MUNICIPALITY" NVARCHAR(200),
    "HYVINVOINTIALUE" NVARCHAR(200),
    PRIMARY KEY ("MUNICIPALITY", "HYVINVOINTIALUE")
)
'''

try:
    cursor.execute(create_municipality_sql)
    print("✓ FI_MUNICIPALITY_MAPPING table created successfully!")
except Exception as e:
    print(f"FI_MUNICIPALITY_MAPPING table: {e}")

✓ FI_MUNICIPALITY_MAPPING table created successfully!


In [25]:
# Create Conscript Organization Table
create_organization_sql = f'''
CREATE COLUMN TABLE "{SCHEMA}"."FI_CONSCRIPT_ORGANIZATION" (
    "CONSCRIPT_ID" INTEGER PRIMARY KEY,
    "GARRISON" NVARCHAR(200),
    "UNIT" NVARCHAR(200),
    "HOME_MUNICIPALITY" NVARCHAR(200),
    "SERVICE_BATCH" NVARCHAR(50)
)
'''

try:
    cursor.execute(create_organization_sql)
    print("✓ FI_CONSCRIPT_ORGANIZATION table created successfully!")
except Exception as e:
    print(f"FI_CONSCRIPT_ORGANIZATION table: {e}")

✓ FI_CONSCRIPT_ORGANIZATION table created successfully!


In [26]:
# Create Conscript Health Data Table (FDF garrison health records)
create_conscript_health_sql = f'''
CREATE COLUMN TABLE "{SCHEMA}"."FI_CONSCRIPT_HEALTH" (
    "CONSCRIPT_ID" INTEGER,
    "DISEASE_CODE" NVARCHAR(20),
    "DISEASE_NAME" NVARCHAR(200),
    "RECORD_DATE" DATE,
    PRIMARY KEY ("CONSCRIPT_ID", "DISEASE_CODE", "RECORD_DATE")
)
'''

try:
    cursor.execute(create_conscript_health_sql)
    print("✓ FI_CONSCRIPT_HEALTH table created successfully!")
except Exception as e:
    print(f"FI_CONSCRIPT_HEALTH table: {e}")

✓ FI_CONSCRIPT_HEALTH table created successfully!


In [27]:
# Create THL Disease Cases Table (regional weekly disease statistics)
create_thl_cases_sql = f'''
CREATE COLUMN TABLE "{SCHEMA}"."FI_THL_DISEASE_CASES" (
    "DISEASE_NAME" NVARCHAR(200),
    "HYVINVOINTIALUE" NVARCHAR(200),
    "YEAR" INTEGER,
    "WEEK" INTEGER,
    "CASE_COUNT" INTEGER,
    PRIMARY KEY ("DISEASE_NAME", "HYVINVOINTIALUE", "YEAR", "WEEK")
)
'''

try:
    cursor.execute(create_thl_cases_sql)
    print("✓ FI_THL_DISEASE_CASES table created successfully!")
except Exception as e:
    print(f"FI_THL_DISEASE_CASES table: {e}")

conn.commit()
print("\n✅ All tables created successfully!")

✓ FI_THL_DISEASE_CASES table created successfully!

✅ All tables created successfully!


## Step 6: Load Data into HANA Tables

In [28]:
# 1. Load Disease Information (Master Data)
disease_info_data = [
    ('J00', 'Rinovirus', 'Rhinovirus', 'Nasal congestion, runny nose, sneezing, sore throat, cough', 
     'Frequent handwashing, avoid close contact with infected individuals', 7, 
     'Direct contact, airborne droplets', 50),
    ('J10', 'Influenssa', 'Influenza', 'Fever, muscle aches, headache, cough, fatigue, sore throat', 
     'Annual vaccination, frequent handwashing, avoid crowds during flu season', 7, 
     'Airborne droplets, direct contact', 30),
    ('U07.1', 'Koronavirus SARS-CoV-2', 'Coronavirus SARS-CoV-2', 'Fever, cough, shortness of breath, loss of taste/smell, fatigue', 
     'Vaccination, mask wearing, social distancing, handwashing', 14, 
     'Airborne transmission, direct contact', 20),
    ('A08.1', 'Norovirus', 'Norovirus', 'Nausea, vomiting, diarrhea, stomach pain, fever', 
     'Thorough handwashing, disinfection of surfaces, food safety', 3, 
     'Contaminated food/water, direct contact, surfaces', 15),
    ('B97.0', 'Adenovirus', 'Adenovirus', 'Fever, sore throat, pink eye, diarrhea, respiratory symptoms', 
     'Frequent handwashing, avoid sharing utensils, avoid touching face', 10, 
     'Direct contact, airborne droplets, contaminated surfaces', 25)
]

insert_disease_sql = f'''
INSERT INTO "{SCHEMA}"."FI_DISEASE_INFO" 
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
'''

inserted_count = 0
for row in disease_info_data:
    try:
        cursor.execute(insert_disease_sql, row)
        inserted_count += 1
    except Exception as e:
        print(f"Error inserting disease info: {e}")

conn.commit()
print(f"✓ Inserted {inserted_count} disease records into FI_DISEASE_INFO")

✓ Inserted 5 disease records into FI_DISEASE_INFO


In [29]:
# 2. Load Garrison Information (Master Data)
garrison_info_data = [
    ('Kainuun prikaati', 'Kajaani', 1200, 'Brigade located in Kainuu region'),
    ('Rannikkoprikaati', 'Turku', 800, 'Coastal defense brigade'),
    ('Karjalan prikaati', 'Kontiolahti', 1500, 'Brigade in Eastern Finland'),
    ('Porin prikaati', 'Pori', 1000, 'Brigade in Western Finland'),
    ('Panssariprikaati', 'Parola', 1300, 'Armored brigade')
]

insert_garrison_sql = f'''
INSERT INTO "{SCHEMA}"."FI_GARRISON_INFO" 
VALUES (?, ?, ?, ?)
'''

inserted_count = 0
for row in garrison_info_data:
    try:
        cursor.execute(insert_garrison_sql, row)
        inserted_count += 1
    except Exception as e:
        print(f"Error inserting garrison info: {e}")

conn.commit()
print(f"✓ Inserted {inserted_count} garrison records into FI_GARRISON_INFO")

✓ Inserted 5 garrison records into FI_GARRISON_INFO


In [30]:
# 3. Load Municipality Mapping
insert_municipality_sql = f'''
INSERT INTO "{SCHEMA}"."FI_MUNICIPALITY_MAPPING" 
VALUES (?, ?)
'''

inserted_count = 0
for _, row in df_municipality_normalized.iterrows():
    try:
        cursor.execute(insert_municipality_sql, (row['municipality'], row['hyvinvointialue']))
        inserted_count += 1
    except Exception as e:
        print(f"Error inserting municipality mapping: {e}")
        if inserted_count == 0:  # Only print first error
            print(f"First error details: {row}")

conn.commit()
print(f"✓ Inserted {inserted_count} municipality mappings into FI_MUNICIPALITY_MAPPING")

✓ Inserted 293 municipality mappings into FI_MUNICIPALITY_MAPPING


In [31]:
# 4. Load Conscript Organization Data
insert_org_sql = f'''
INSERT INTO "{SCHEMA}"."FI_CONSCRIPT_ORGANIZATION" 
VALUES (?, ?, ?, ?, ?)
'''

inserted_count = 0
for _, row in df_organization.iterrows():
    try:
        cursor.execute(insert_org_sql, (
            int(row['ID']),
            row['Joukko-osasto'],
            row['Joukkoyksikkö'],
            row['Kotikunta'],
            row['Saapumiserä']
        ))
        inserted_count += 1
    except Exception as e:
        if inserted_count == 0:
            print(f"Error inserting organization: {e}")

conn.commit()
print(f"✓ Inserted {inserted_count} conscript organization records into FI_CONSCRIPT_ORGANIZATION")

✓ Inserted 3000 conscript organization records into FI_CONSCRIPT_ORGANIZATION


In [32]:
# 5. Load Conscript Health Data
# Generate random dates for health records (simulating 2025-2026 data)
import random
from datetime import datetime, timedelta

insert_health_sql = f'''
INSERT INTO "{SCHEMA}"."FI_CONSCRIPT_HEALTH" 
VALUES (?, ?, ?, ?)
'''

inserted_count = 0
start_date = datetime(2025, 1, 1)
end_date = datetime(2026, 1, 28)

for _, row in df_fdf_health.iterrows():
    try:
        # Generate random date within range
        days_between = (end_date - start_date).days
        random_days = random.randint(0, days_between)
        record_date = start_date + timedelta(days=random_days)
        
        cursor.execute(insert_health_sql, (
            int(row['ID']),
            row['Tautiluokitus'],
            row['Taudin nimi'],
            record_date.date()
        ))
        inserted_count += 1
    except Exception as e:
        if inserted_count == 0:
            print(f"Error inserting health record: {e}")

conn.commit()
print(f"✓ Inserted {inserted_count} conscript health records into FI_CONSCRIPT_HEALTH")

✓ Inserted 3000 conscript health records into FI_CONSCRIPT_HEALTH


In [33]:
# 6. Load THL Disease Cases (combine 2025 and 2026 data)
# Note: The THL data needs better parsing - for now, insert sample aggregated data

# Combine both years
df_thl_combined = pd.concat([df_thl_2025_transformed, df_thl_2026_transformed], ignore_index=True)

insert_thl_sql = f'''
INSERT INTO "{SCHEMA}"."FI_THL_DISEASE_CASES" 
VALUES (?, ?, ?, ?, ?)
'''

inserted_count = 0
error_count = 0

for _, row in df_thl_combined.iterrows():
    try:
        # Clean and validate data
        case_count = int(row['case_count']) if pd.notna(row['case_count']) and str(row['case_count']).replace('.', '').replace('-', '').isdigit() else 0
        
        cursor.execute(insert_thl_sql, (
            str(row['disease_name'])[:200],
            str(row['hyvinvointialue'])[:200],
            int(row['year']),
            int(row['week']),
            case_count
        ))
        inserted_count += 1
    except Exception as e:
        error_count += 1
        if error_count <= 3:
            print(f"Error inserting THL case: {e}, Row: {row.to_dict()}")

conn.commit()
print(f"✓ Inserted {inserted_count} THL disease case records into FI_THL_DISEASE_CASES")
print(f"  Errors: {error_count}")

Error inserting THL case: (301, 'unique constraint violated: Table(FI_THL_DISEASE_CASES), Index(_SYS_TREE_CS_#271038_#0_#P0) with error: unique constraint violation for table H00::HACKATHON_USR:FI_THL_DISEASE_CASES$delta_1$en, key=31302c496e666c75656e7373613b33302c4974c3a42d557564656e6d61616e20687976696e766f696e7469616c75653b323032353b31 already exists as udiv=1; indexname=_SYS_TREE_CS_#271038_#0_#P0'), Row: {'disease_name': 'Influenssa', 'hyvinvointialue': 'Itä-Uudenmaan hyvinvointialue', 'year': 2025, 'week': 1, 'case_count': 16}
Error inserting THL case: (301, 'unique constraint violated: Table(FI_THL_DISEASE_CASES), Index(_SYS_TREE_CS_#271038_#0_#P0) with error: unique constraint violation for table H00::HACKATHON_USR:FI_THL_DISEASE_CASES$delta_1$en, key=31302c496e666c75656e7373613b33312c4b65736b692d557564656e6d61616e20687976696e766f696e7469616c75653b323032353b31 already exists as udiv=2; indexname=_SYS_TREE_CS_#271038_#0_#P0'), Row: {'disease_name': 'Influenssa', 'hyvinvointialue'

## Step 7: Add Geolocation Data for Municipalities

Geocode Finnish municipalities using geopy to get latitude/longitude coordinates

In [10]:
# Install and import geopy for geocoding
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import time

# Initialize geocoder with user agent
geolocator = Nominatim(user_agent="finland_garrison_health_monitor")

# Get unique municipalities
unique_municipalities = df_municipality_normalized['municipality'].unique()
print(f"Total unique municipalities to geocode: {len(unique_municipalities)}")

# Geocode each municipality
municipality_coordinates = []
failed_geocodes = []

for idx, municipality in enumerate(unique_municipalities):
    try:
        # Add "Finland" to improve geocoding accuracy
        location = geolocator.geocode(f"{municipality}, Finland", timeout=10)
        
        if location:
            municipality_coordinates.append({
                'municipality': municipality,
                'latitude': location.latitude,
                'longitude': location.longitude
            })
            print(f"{idx+1}/{len(unique_municipalities)}: ✓ {municipality} - ({location.latitude:.4f}, {location.longitude:.4f})")
        else:
            failed_geocodes.append(municipality)
            print(f"{idx+1}/{len(unique_municipalities)}: ✗ {municipality} - NOT FOUND")
        
        # Be respectful to the geocoding service - add delay
        time.sleep(1)
        
    except (GeocoderTimedOut, GeocoderServiceError) as e:
        failed_geocodes.append(municipality)
        print(f"{idx+1}/{len(unique_municipalities)}: ✗ {municipality} - ERROR: {e}")
        time.sleep(2)  # Longer delay after error
    except Exception as e:
        failed_geocodes.append(municipality)
        print(f"{idx+1}/{len(unique_municipalities)}: ✗ {municipality} - UNEXPECTED ERROR: {e}")

# Create DataFrame with coordinates
df_municipality_coords = pd.DataFrame(municipality_coordinates)

print(f"\n{'='*80}")
print(f"✅ Geocoding complete!")
print(f"  • Successfully geocoded: {len(municipality_coordinates)} municipalities")
print(f"  • Failed to geocode: {len(failed_geocodes)} municipalities")
if failed_geocodes:
    print(f"  • Failed municipalities: {', '.join(failed_geocodes[:10])}{'...' if len(failed_geocodes) > 10 else ''}")
print(f"{'='*80}\n")

df_municipality_coords.head(10)

Total unique municipalities to geocode: 293
1/293: ✓ Imatra - (61.1959, 28.7767)
2/293: ✓ Lappeenranta - (61.0583, 28.1862)
3/293: ✓ Lemi - (61.0616, 27.8042)
4/293: ✓ Luumäki - (60.9225, 27.5693)
5/293: ✓ Parikkala - (61.5580, 29.5014)
6/293: ✓ Rautjärvi - (61.3648, 29.2090)
7/293: ✓ Ruokolahti - (61.2925, 28.8121)
8/293: ✓ Savitaipale - (61.1977, 27.6827)
9/293: ✓ Taipalsaari - (61.1599, 28.0604)
10/293: ✓ Alajärvi - (62.9999, 23.8168)
11/293: ✓ Alavus - (62.5862, 23.6185)
12/293: ✓ Evijärvi - (63.3671, 23.4769)
13/293: ✓ Ilmajoki - (62.7313, 22.5798)
14/293: ✓ Isojoki - (62.1143, 21.9588)
15/293: ✓ Isokyrö - (63.0000, 22.3167)
16/293: ✓ Karijoki - (62.3075, 21.7078)
17/293: ✓ Kauhajoki - (62.4317, 22.1842)
18/293: ✓ Kauhava - (63.0994, 23.0570)
19/293: ✓ Kuortane - (62.8070, 23.5069)
20/293: ✓ Kurikka - (62.6172, 22.3992)
21/293: ✓ Lappajärvi - (63.2193, 23.6284)
22/293: ✓ Lapua - (62.9703, 23.0069)
23/293: ✓ Seinäjoki - (62.7907, 22.8398)
24/293: ✓ Soini - (62.8738, 24.2077)
25/293

,municipality,latitude,longitude
0,Imatra,61.195926,28.776659
1,Lappeenranta,61.058342,28.186229
2,Lemi,61.061552,27.804161
3,Luumäki,60.922520,27.569310
4,Parikkala,61.557966,29.501357
5,Rautjärvi,61.364832,29.209014
6,Ruokolahti,61.292538,28.812060
7,Savitaipale,61.197725,27.682650
8,Taipalsaari,61.159937,28.060422
9,Alajärvi,62.999864,23.816761


In [12]:
import os
from dotenv import load_dotenv
from hdbcli import dbapi

# Load environment variables
load_dotenv()

HANA_HOST = os.getenv("HANA_HOST")
HANA_PORT = os.getenv("HANA_PORT")
HANA_USER = os.getenv("HANA_HACKATHON_USER")
HANA_PASSWORD = os.getenv("HANA_HACKATHON_PASSWORD")

# Connect to HANA
conn = dbapi.connect(
    address=HANA_HOST,
    port=HANA_PORT,
    user="HACKATHON_USR",
    password=HANA_PASSWORD
)

cursor = conn.cursor()
print("Connected to HANA successfully!")

# Use existing schema
SCHEMA = "HACKATHON_USR"
print(f"Using schema: {SCHEMA}")

# Create HANA table for municipality coordinates
create_coords_sql = f'''
CREATE COLUMN TABLE "{SCHEMA}"."FI_MUNICIPALITY_COORDINATES" (
    "MUNICIPALITY" NVARCHAR(200) PRIMARY KEY,
    "LATITUDE" DOUBLE,
    "LONGITUDE" DOUBLE
)
'''

try:
    # Drop table if exists
    cursor.execute(f'DROP TABLE "{SCHEMA}"."FI_MUNICIPALITY_COORDINATES"')
    print("Dropped existing FI_MUNICIPALITY_COORDINATES table")
except Exception as e:
    print(f"Table does not exist yet: {e}")

try:
    cursor.execute(create_coords_sql)
    print("✓ FI_MUNICIPALITY_COORDINATES table created successfully!")
except Exception as e:
    print(f"Error creating table: {e}")

Connected to HANA successfully!
Using schema: HACKATHON_USR
Table does not exist yet: (259, 'invalid table name: FI_MUNICIPALITY_COORDINATES: line 1 col 28 (at pos 27)')
✓ FI_MUNICIPALITY_COORDINATES table created successfully!


In [13]:
# Load municipality coordinates into HANA
insert_coords_sql = f'''
INSERT INTO "{SCHEMA}"."FI_MUNICIPALITY_COORDINATES" 
VALUES (?, ?, ?)
'''

inserted_count = 0
for _, row in df_municipality_coords.iterrows():
    try:
        cursor.execute(insert_coords_sql, (
            row['municipality'],
            float(row['latitude']),
            float(row['longitude'])
        ))
        inserted_count += 1
    except Exception as e:
        if inserted_count == 0:
            print(f"Error inserting coordinates: {e}")

conn.commit()
print(f"✓ Inserted {inserted_count} municipality coordinates into FI_MUNICIPALITY_COORDINATES")

# Verify data
cursor.execute(f'SELECT COUNT(*) FROM "{SCHEMA}"."FI_MUNICIPALITY_COORDINATES"')
count = cursor.fetchone()[0]
print(f"  • Total rows in FI_MUNICIPALITY_COORDINATES: {count}")

# Show sample data
cursor.execute(f'SELECT * FROM "{SCHEMA}"."FI_MUNICIPALITY_COORDINATES" LIMIT 5')
sample_rows = cursor.fetchall()
print(f"\nSample coordinates:")
for row in sample_rows:
    print(f"  {row[0]}: ({row[1]:.4f}, {row[2]:.4f})")

✓ Inserted 293 municipality coordinates into FI_MUNICIPALITY_COORDINATES
  • Total rows in FI_MUNICIPALITY_COORDINATES: 293

Sample coordinates:
  Imatra: (61.1959, 28.7767)
  Lappeenranta: (61.0583, 28.1862)
  Lemi: (61.0616, 27.8042)
  Luumäki: (60.9225, 27.5693)
  Parikkala: (61.5580, 29.5014)


In [14]:
print("\n" + "="*80)
print("✅ DATA LOADING COMPLETE!")
print("="*80)
print("\nSummary of loaded data:")
print(f"  • Disease Information: 5 diseases")
print(f"  • Garrison Information: 5 garrisons")
print(f"  • Municipality Mappings: {df_municipality_normalized.shape[0]} mappings")
print(f"  • Conscript Organization: {df_organization.shape[0]} conscripts")
print(f"  • Conscript Health Records: {df_fdf_health.shape[0]} health records")
print(f"  • THL Disease Cases: Regional weekly statistics")
print("\nAll data has been successfully persisted to HANA schema:", SCHEMA)


✅ DATA LOADING COMPLETE!

Summary of loaded data:
  • Disease Information: 5 diseases
  • Garrison Information: 5 garrisons
  • Municipality Mappings: 293 mappings
  • Conscript Organization: 3000 conscripts
  • Conscript Health Records: 3000 health records
  • THL Disease Cases: Regional weekly statistics

All data has been successfully persisted to HANA schema: HACKATHON_USR


## Data Model Documentation for RAG

### Overview
This data model supports epidemic monitoring and prediction for Finnish Defense Forces garrisons. It enables analysis of disease trends, risk assessment, and countermeasure planning.

### Schema: HACKATHON_USR

---

### Tables

#### 1. **FI_DISEASE_INFO** (Master Data)
Disease catalog with medical information for epidemic monitoring.

**Columns:**
- `DISEASE_CODE` (NVARCHAR(20), PK): ICD-10 disease classification code
- `DISEASE_NAME_FI` (NVARCHAR(200)): Finnish disease name
- `DISEASE_NAME_EN` (NVARCHAR(200)): English disease name
- `SYMPTOMS` (NVARCHAR(2000)): Common symptoms description
- `PREVENTION` (NVARCHAR(2000)): Prevention measures
- `AVERAGE_DURATION_DAYS` (INTEGER): Typical illness duration
- `HOW_IT_SPREADS` (NVARCHAR(1000)): Transmission methods
- `EPIDEMIC_THRESHOLD` (INTEGER): Case count threshold for epidemic alert

**Use Cases:**
- Get disease information for agent responses
- Determine epidemic thresholds
- Provide prevention recommendations

---

#### 2. **FI_GARRISON_INFO** (Master Data)
Information about Finnish Defense Forces garrisons.

**Columns:**
- `GARRISON_NAME` (NVARCHAR(200), PK): Official garrison name
- `LOCATION` (NVARCHAR(200)): City/municipality location
- `CAPACITY` (INTEGER): Maximum conscript capacity
- `DESCRIPTION` (NVARCHAR(1000)): Additional garrison details

**Use Cases:**
- Identify garrison locations
- Calculate epidemic impact based on capacity
- Regional disease analysis

---

#### 3. **FI_MUNICIPALITY_MAPPING**
Maps municipalities to wellbeing service counties (hyvinvointialue).

**Columns:**
- `MUNICIPALITY` (NVARCHAR(200), PK1): Municipality name
- `HYVINVOINTIALUE` (NVARCHAR(200), PK2): Wellbeing services county

**Use Cases:**
- Link conscript home locations to regional disease data
- Weekend travel risk assessment
- Regional disease tracking

---

#### 4. **FI_CONSCRIPT_ORGANIZATION**
Organizational assignment and home location of conscripts.

**Columns:**
- `CONSCRIPT_ID` (INTEGER, PK): Unique conscript identifier
- `GARRISON` (NVARCHAR(200)): Assigned garrison (Joukko-osasto)
- `UNIT` (NVARCHAR(200)): Assigned unit (Joukkoyksikkö)
- `HOME_MUNICIPALITY` (NVARCHAR(200)): Home municipality (Kotikunta)
- `SERVICE_BATCH` (NVARCHAR(50)): Service intake batch (Saapumiserä)

**Use Cases:**
- Track conscript assignments by garrison
- Analyze disease spread patterns by unit
- Weekend travel risk based on home municipality disease rates

---

#### 5. **FI_CONSCRIPT_HEALTH**
Individual conscript health records (garrison-level disease tracking).

**Columns:**
- `CONSCRIPT_ID` (INTEGER, PK1): References FI_CONSCRIPT_ORGANIZATION
- `DISEASE_CODE` (NVARCHAR(20), PK2): ICD-10 code, references FI_DISEASE_INFO
- `DISEASE_NAME` (NVARCHAR(200)): Disease name for readability
- `RECORD_DATE` (DATE, PK3): Date of diagnosis/record

**Use Cases:**
- Real-time garrison disease surveillance
- Identify outbreak patterns within units
- Calculate infection rates per garrison
- Trend analysis over time

---

#### 6. **FI_THL_DISEASE_CASES**
National THL (Finnish Institute for Health and Welfare) regional weekly disease statistics.

**Columns:**
- `DISEASE_NAME` (NVARCHAR(200), PK1): Disease name
- `HYVINVOINTIALUE` (NVARCHAR(200), PK2): Wellbeing services county
- `YEAR` (INTEGER, PK3): Year of the week
- `WEEK` (INTEGER, PK4): ISO week number
- `CASE_COUNT` (INTEGER): Number of reported cases

**Use Cases:**
- Regional disease trend analysis
- Weekend travel risk assessment (conscripts visiting home)
- Early warning system based on home region outbreaks
- Seasonal pattern recognition

---

### Key Relationships

```
FI_CONSCRIPT_ORGANIZATION
  ├─ CONSCRIPT_ID → FI_CONSCRIPT_HEALTH.CONSCRIPT_ID
  └─ HOME_MUNICIPALITY → FI_MUNICIPALITY_MAPPING.MUNICIPALITY
       └─ HYVINVOINTIALUE → FI_THL_DISEASE_CASES.HYVINVOINTIALUE

FI_CONSCRIPT_HEALTH
  └─ DISEASE_CODE → FI_DISEASE_INFO.DISEASE_CODE

FI_THL_DISEASE_CASES
  ├─ DISEASE_NAME (mapped to) FI_DISEASE_INFO.DISEASE_NAME_FI
  └─ HYVINVOINTIALUE → FI_MUNICIPALITY_MAPPING.HYVINVOINTIALUE
```

---

### Sample RAG Queries

**1. Epidemic Detection**
```sql
-- Check if garrison has potential epidemic
SELECT 
    g.GARRISON,
    ch.DISEASE_NAME,
    COUNT(DISTINCT ch.CONSCRIPT_ID) as cases,
    di.EPIDEMIC_THRESHOLD,
    CASE WHEN COUNT(DISTINCT ch.CONSCRIPT_ID) >= di.EPIDEMIC_THRESHOLD 
         THEN 'EPIDEMIC ALERT' 
         ELSE 'NORMAL' 
    END as status
FROM "HACKATHON_USR"."FI_CONSCRIPT_HEALTH" ch
JOIN "HACKATHON_USR"."FI_CONSCRIPT_ORGANIZATION" co 
    ON ch.CONSCRIPT_ID = co.CONSCRIPT_ID
JOIN "HACKATHON_USR"."FI_DISEASE_INFO" di 
    ON ch.DISEASE_CODE = di.DISEASE_CODE
WHERE ch.RECORD_DATE >= ADD_DAYS(CURRENT_DATE, -14)
GROUP BY g.GARRISON, ch.DISEASE_NAME, di.EPIDEMIC_THRESHOLD
HAVING COUNT(DISTINCT ch.CONSCRIPT_ID) >= di.EPIDEMIC_THRESHOLD;
```

**2. Weekend Travel Risk**
```sql
-- Assess disease risk in conscript home regions
SELECT 
    co.GARRISON,
    co.HOME_MUNICIPALITY,
    mm.HYVINVOINTIALUE,
    thl.DISEASE_NAME,
    thl.CASE_COUNT,
    thl.YEAR,
    thl.WEEK
FROM "HACKATHON_USR"."FI_CONSCRIPT_ORGANIZATION" co
JOIN "HACKATHON_USR"."FI_MUNICIPALITY_MAPPING" mm 
    ON co.HOME_MUNICIPALITY = mm.MUNICIPALITY
JOIN "HACKATHON_USR"."FI_THL_DISEASE_CASES" thl 
    ON mm.HYVINVOINTIALUE = thl.HYVINVOINTIALUE
WHERE thl.YEAR = 2026 AND thl.WEEK >= 1
ORDER BY thl.CASE_COUNT DESC;
```

**3. Disease Trends by Garrison**
```sql
-- Analyze disease trends in specific garrison
SELECT 
    WEEK(ch.RECORD_DATE) as week,
    ch.DISEASE_NAME,
    COUNT(DISTINCT ch.CONSCRIPT_ID) as cases,
    di.PREVENTION
FROM "HACKATHON_USR"."FI_CONSCRIPT_HEALTH" ch
JOIN "HACKATHON_USR"."FI_CONSCRIPT_ORGANIZATION" co 
    ON ch.CONSCRIPT_ID = co.CONSCRIPT_ID
JOIN "HACKATHON_USR"."FI_DISEASE_INFO" di 
    ON ch.DISEASE_CODE = di.DISEASE_CODE
WHERE co.GARRISON = 'Kainuun prikaati'
  AND ch.RECORD_DATE >= ADD_DAYS(CURRENT_DATE, -90)
GROUP BY WEEK(ch.RECORD_DATE), ch.DISEASE_NAME, di.PREVENTION
ORDER BY week DESC, cases DESC;
```

**4. Get Disease Information and Countermeasures**
```sql
-- Retrieve disease info for agent recommendations
SELECT 
    DISEASE_NAME_EN,
    SYMPTOMS,
    PREVENTION,
    AVERAGE_DURATION_DAYS,
    HOW_IT_SPREADS
FROM "HACKATHON_USR"."FI_DISEASE_INFO"
WHERE DISEASE_CODE = 'J10'; -- Influenza
```

**5. Impact on Training Assessment**
```sql
-- Calculate percentage of conscripts affected by disease
SELECT 
    co.GARRISON,
    gi.CAPACITY,
    COUNT(DISTINCT co.CONSCRIPT_ID) as total_conscripts,
    COUNT(DISTINCT ch.CONSCRIPT_ID) as sick_conscripts,
    ROUND(COUNT(DISTINCT ch.CONSCRIPT_ID) * 100.0 / COUNT(DISTINCT co.CONSCRIPT_ID), 2) as sick_percentage
FROM "HACKATHON_USR"."FI_CONSCRIPT_ORGANIZATION" co
LEFT JOIN "HACKATHON_USR"."FI_CONSCRIPT_HEALTH" ch 
    ON co.CONSCRIPT_ID = ch.CONSCRIPT_ID 
    AND ch.RECORD_DATE >= ADD_DAYS(CURRENT_DATE, -7)
JOIN "HACKATHON_USR"."FI_GARRISON_INFO" gi 
    ON co.GARRISON = gi.GARRISON_NAME
GROUP BY co.GARRISON, gi.CAPACITY;
```

---

### Agent Use Case Support

**Question: "Is there a possibility for an epidemic?"**
- Query FI_CONSCRIPT_HEALTH for recent cases
- Compare against FI_DISEASE_INFO.EPIDEMIC_THRESHOLD
- Check FI_THL_DISEASE_CASES for regional trends

**Question: "What kind of epidemic?"**
- Join FI_CONSCRIPT_HEALTH with FI_DISEASE_INFO
- Return disease name, symptoms, transmission method

**Question: "What countermeasures could be taken?"**
- Retrieve FI_DISEASE_INFO.PREVENTION
- Calculate affected conscript percentage
- Suggest quarantine plans based on unit assignments

**Question: "How could it affect training?"**
- Calculate sick percentage per garrison/unit
- Use FI_DISEASE_INFO.AVERAGE_DURATION_DAYS for timeline
- Project training capacity impact

**Question: "Are there disease trends in specific areas?"**
- Analyze FI_THL_DISEASE_CASES by hyvinvointialue
- Cross-reference with FI_CONSCRIPT_ORGANIZATION.HOME_MUNICIPALITY
- Identify high-risk home regions for weekend travel

**Question: "Do some infectious diseases trend in some garrisons?"**
- Time-series analysis of FI_CONSCRIPT_HEALTH by garrison
- Compare with national trends from FI_THL_DISEASE_CASES
- Identify garrison-specific outbreak patterns

## Next Steps

To use this data model with a RAG-based AI agent:

1. **Configure AI Agent** with access to HACKATHON_USR schema
2. **Provide Schema Context** from the documentation above
3. **Enable SQL Generation** for natural language queries about:
   - Current garrison health status
   - Epidemic risk assessment
   - Disease trends and patterns
   - Countermeasure recommendations
   - Training impact calculations

4. **Enhance with Additional Data** (optional):
   - Historical epidemic thresholds per garrison
   - Weather data (can affect disease spread)
   - Training schedules for impact assessment
   - Hospital capacity data

The data model is now ready for RAG-based querying!

## Data Model for RAG (Python Format)

This cell contains the complete data model in Python format for easy import into RAG systems.

In [ ]:
# Finnish Garrison Health Data Model for RAG
# Schema: HACKATHON_USR
# Use Case: Epidemic monitoring and prediction for Finnish Defense Forces

FINLAND_HEALTH_DATA_MODEL = {
    "schema": "HACKATHON_USR",
    "description": "Epidemic monitoring and prediction system for Finnish Defense Forces garrisons. Tracks conscript health, regional disease trends, and enables risk assessment for garrison-wide epidemics.",
    
    "tables": {
        "FI_DISEASE_INFO": {
            "description": "Master data table containing disease information for epidemic monitoring",
            "type": "master_data",
            "columns": {
                "DISEASE_CODE": {
                    "type": "NVARCHAR(20)",
                    "primary_key": True,
                    "description": "ICD-10 disease classification code (e.g., J00 for Rhinovirus, J10 for Influenza)"
                },
                "DISEASE_NAME_FI": {
                    "type": "NVARCHAR(200)",
                    "description": "Finnish disease name (e.g., Rinovirus, Influenssa)"
                },
                "DISEASE_NAME_EN": {
                    "type": "NVARCHAR(200)",
                    "description": "English disease name"
                },
                "SYMPTOMS": {
                    "type": "NVARCHAR(2000)",
                    "description": "Common symptoms description"
                },
                "PREVENTION": {
                    "type": "NVARCHAR(2000)",
                    "description": "Prevention measures and recommendations"
                },
                "AVERAGE_DURATION_DAYS": {
                    "type": "INTEGER",
                    "description": "Typical illness duration in days"
                },
                "HOW_IT_SPREADS": {
                    "type": "NVARCHAR(1000)",
                    "description": "Transmission methods (e.g., airborne, contact, surfaces)"
                },
                "EPIDEMIC_THRESHOLD": {
                    "type": "INTEGER",
                    "description": "Case count threshold for epidemic alert in a garrison"
                }
            },
            "sample_data": [
                "J00 - Rinovirus - 50 case threshold",
                "J10 - Influenssa - 30 case threshold",
                "U07.1 - Koronavirus - 20 case threshold",
                "A08.1 - Norovirus - 15 case threshold",
                "B97.0 - Adenovirus - 25 case threshold"
            ]
        },
        
        "FI_GARRISON_INFO": {
            "description": "Information about Finnish Defense Forces garrisons",
            "type": "master_data",
            "columns": {
                "GARRISON_NAME": {
                    "type": "NVARCHAR(200)",
                    "primary_key": True,
                    "description": "Official garrison name (e.g., Kainuun prikaati, Rannikkoprikaati)"
                },
                "LOCATION": {
                    "type": "NVARCHAR(200)",
                    "description": "City or municipality location"
                },
                "CAPACITY": {
                    "type": "INTEGER",
                    "description": "Maximum conscript capacity"
                },
                "DESCRIPTION": {
                    "type": "NVARCHAR(1000)",
                    "description": "Additional garrison details"
                }
            },
            "sample_data": [
                "Kainuun prikaati - Kajaani - 1200 capacity",
                "Rannikkoprikaati - Turku - 800 capacity",
                "Karjalan prikaati - Kontiolahti - 1500 capacity"
            ]
        },
        
        "FI_MUNICIPALITY_MAPPING": {
            "description": "Maps municipalities to wellbeing service counties (hyvinvointialue) for regional disease tracking",
            "type": "reference_data",
            "columns": {
                "MUNICIPALITY": {
                    "type": "NVARCHAR(200)",
                    "primary_key": True,
                    "description": "Municipality name (kunta)"
                },
                "HYVINVOINTIALUE": {
                    "type": "NVARCHAR(200)",
                    "primary_key": True,
                    "description": "Wellbeing services county name"
                }
            },
            "purpose": "Links conscript home locations to regional THL disease statistics for weekend travel risk assessment"
        },
        
        "FI_MUNICIPALITY_COORDINATES": {
            "description": "Geographic coordinates (latitude/longitude) for Finnish municipalities",
            "type": "reference_data",
            "columns": {
                "MUNICIPALITY": {
                    "type": "NVARCHAR(200)",
                    "primary_key": True,
                    "description": "Municipality name (kunta)",
                    "references": "FI_MUNICIPALITY_MAPPING.MUNICIPALITY"
                },
                "LATITUDE": {
                    "type": "DOUBLE",
                    "description": "Latitude coordinate in decimal degrees"
                },
                "LONGITUDE": {
                    "type": "DOUBLE",
                    "description": "Longitude coordinate in decimal degrees"
                }
            },
            "purpose": "Enable spatial analysis, mapping, and distance-based queries for epidemic risk assessment",
            "use_cases": [
                "Map disease outbreaks geographically",
                "Calculate distance between garrisons and high-risk regions",
                "Visualize disease spread patterns on maps",
                "Identify geographic clusters of disease cases"
            ]
        },
        
        "FI_CONSCRIPT_ORGANIZATION": {
            "description": "Organizational assignment and home location of each conscript",
            "type": "transactional",
            "columns": {
                "CONSCRIPT_ID": {
                    "type": "INTEGER",
                    "primary_key": True,
                    "description": "Unique conscript identifier"
                },
                "GARRISON": {
                    "type": "NVARCHAR(200)",
                    "description": "Assigned garrison (Joukko-osasto)",
                    "references": "FI_GARRISON_INFO.GARRISON_NAME"
                },
                "UNIT": {
                    "type": "NVARCHAR(200)",
                    "description": "Assigned unit within garrison (Joukkoyksikkö)"
                },
                "HOME_MUNICIPALITY": {
                    "type": "NVARCHAR(200)",
                    "description": "Conscript's home municipality (Kotikunta)",
                    "references": "FI_MUNICIPALITY_MAPPING.MUNICIPALITY"
                },
                "SERVICE_BATCH": {
                    "type": "NVARCHAR(50)",
                    "description": "Service intake batch (Saapumiserä, e.g., 1/26, 2/25)"
                }
            },
            "row_count": 3000,
            "use_cases": [
                "Track conscript assignments by garrison",
                "Analyze disease spread patterns by unit",
                "Weekend travel risk based on home municipality disease rates"
            ]
        },
        
        "FI_CONSCRIPT_HEALTH": {
            "description": "Individual conscript health records - garrison-level disease tracking",
            "type": "transactional",
            "columns": {
                "CONSCRIPT_ID": {
                    "type": "INTEGER",
                    "primary_key": True,
                    "description": "Reference to conscript",
                    "references": "FI_CONSCRIPT_ORGANIZATION.CONSCRIPT_ID"
                },
                "DISEASE_CODE": {
                    "type": "NVARCHAR(20)",
                    "primary_key": True,
                    "description": "ICD-10 disease code",
                    "references": "FI_DISEASE_INFO.DISEASE_CODE"
                },
                "DISEASE_NAME": {
                    "type": "NVARCHAR(200)",
                    "description": "Disease name for readability"
                },
                "RECORD_DATE": {
                    "type": "DATE",
                    "primary_key": True,
                    "description": "Date of diagnosis or health record"
                }
            },
            "row_count": 3000,
            "use_cases": [
                "Real-time garrison disease surveillance",
                "Identify outbreak patterns within units",
                "Calculate infection rates per garrison",
                "Time-series trend analysis"
            ]
        },
        
        "FI_THL_DISEASE_CASES": {
            "description": "National THL (Finnish Institute for Health and Welfare) regional weekly disease statistics",
            "type": "transactional",
            "columns": {
                "DISEASE_NAME": {
                    "type": "NVARCHAR(200)",
                    "primary_key": True,
                    "description": "Disease name in Finnish"
                },
                "HYVINVOINTIALUE": {
                    "type": "NVARCHAR(200)",
                    "primary_key": True,
                    "description": "Wellbeing services county",
                    "references": "FI_MUNICIPALITY_MAPPING.HYVINVOINTIALUE"
                },
                "YEAR": {
                    "type": "INTEGER",
                    "primary_key": True,
                    "description": "Year of the reporting week"
                },
                "WEEK": {
                    "type": "INTEGER",
                    "primary_key": True,
                    "description": "ISO week number (1-52/53)"
                },
                "CASE_COUNT": {
                    "type": "INTEGER",
                    "description": "Number of reported disease cases in the region for that week"
                }
            },
            "data_range": "2025-2026",
            "use_cases": [
                "Regional disease trend analysis",
                "Weekend travel risk assessment (conscripts visiting home)",
                "Early warning system based on home region outbreaks",
                "Seasonal pattern recognition"
            ]
        }
    },
    
    "relationships": [
        {
            "from": "FI_CONSCRIPT_ORGANIZATION.CONSCRIPT_ID",
            "to": "FI_CONSCRIPT_HEALTH.CONSCRIPT_ID",
            "type": "one_to_many",
            "description": "One conscript can have multiple health records"
        },
        {
            "from": "FI_CONSCRIPT_ORGANIZATION.HOME_MUNICIPALITY",
            "to": "FI_MUNICIPALITY_MAPPING.MUNICIPALITY",
            "type": "many_to_one",
            "description": "Links conscript home to wellbeing county"
        },
        {
            "from": "FI_MUNICIPALITY_MAPPING.MUNICIPALITY",
            "to": "FI_MUNICIPALITY_COORDINATES.MUNICIPALITY",
            "type": "one_to_one",
            "description": "Each municipality has geographic coordinates"
        },
        {
            "from": "FI_MUNICIPALITY_MAPPING.HYVINVOINTIALUE",
            "to": "FI_THL_DISEASE_CASES.HYVINVOINTIALUE",
            "type": "one_to_many",
            "description": "Regional disease statistics per wellbeing county"
        },
        {
            "from": "FI_CONSCRIPT_HEALTH.DISEASE_CODE",
            "to": "FI_DISEASE_INFO.DISEASE_CODE",
            "type": "many_to_one",
            "description": "Links health records to disease information"
        },
        {
            "from": "FI_CONSCRIPT_ORGANIZATION.GARRISON",
            "to": "FI_GARRISON_INFO.GARRISON_NAME",
            "type": "many_to_one",
            "description": "Links conscripts to their garrison"
        }
    ],
    
    "common_queries": {
        "epidemic_detection": {
            "description": "Check if a garrison has a potential epidemic",
            "sql": '''
SELECT 
    co.GARRISON,
    ch.DISEASE_NAME,
    COUNT(DISTINCT ch.CONSCRIPT_ID) as current_cases,
    di.EPIDEMIC_THRESHOLD,
    CASE WHEN COUNT(DISTINCT ch.CONSCRIPT_ID) >= di.EPIDEMIC_THRESHOLD 
         THEN 'EPIDEMIC ALERT' 
         ELSE 'NORMAL' 
    END as status,
    di.PREVENTION as recommended_measures
FROM "HACKATHON_USR"."FI_CONSCRIPT_HEALTH" ch
JOIN "HACKATHON_USR"."FI_CONSCRIPT_ORGANIZATION" co 
    ON ch.CONSCRIPT_ID = co.CONSCRIPT_ID
JOIN "HACKATHON_USR"."FI_DISEASE_INFO" di 
    ON ch.DISEASE_CODE = di.DISEASE_CODE
WHERE ch.RECORD_DATE >= ADD_DAYS(CURRENT_DATE, -14)
GROUP BY co.GARRISON, ch.DISEASE_NAME, di.EPIDEMIC_THRESHOLD, di.PREVENTION
HAVING COUNT(DISTINCT ch.CONSCRIPT_ID) >= di.EPIDEMIC_THRESHOLD
'''
        },
        
        "weekend_travel_risk": {
            "description": "Assess disease risk in conscript home regions for weekend visits",
            "sql": '''
SELECT 
    co.GARRISON,
    co.HOME_MUNICIPALITY,
    mm.HYVINVOINTIALUE,
    thl.DISEASE_NAME,
    thl.CASE_COUNT,
    thl.YEAR,
    thl.WEEK,
    COUNT(DISTINCT co.CONSCRIPT_ID) as conscripts_from_region
FROM "HACKATHON_USR"."FI_CONSCRIPT_ORGANIZATION" co
JOIN "HACKATHON_USR"."FI_MUNICIPALITY_MAPPING" mm 
    ON co.HOME_MUNICIPALITY = mm.MUNICIPALITY
JOIN "HACKATHON_USR"."FI_THL_DISEASE_CASES" thl 
    ON mm.HYVINVOINTIALUE = thl.HYVINVOINTIALUE
WHERE thl.YEAR = 2026 AND thl.WEEK >= WEEK(CURRENT_DATE)
GROUP BY co.GARRISON, co.HOME_MUNICIPALITY, mm.HYVINVOINTIALUE, 
         thl.DISEASE_NAME, thl.CASE_COUNT, thl.YEAR, thl.WEEK
ORDER BY thl.CASE_COUNT DESC
'''
        },
        
        "garrison_disease_trends": {
            "description": "Analyze disease trends over time in a specific garrison",
            "sql": '''
SELECT 
    WEEK(ch.RECORD_DATE) as week,
    YEAR(ch.RECORD_DATE) as year,
    ch.DISEASE_NAME,
    COUNT(DISTINCT ch.CONSCRIPT_ID) as cases,
    di.AVERAGE_DURATION_DAYS,
    di.PREVENTION
FROM "HACKATHON_USR"."FI_CONSCRIPT_HEALTH" ch
JOIN "HACKATHON_USR"."FI_CONSCRIPT_ORGANIZATION" co 
    ON ch.CONSCRIPT_ID = co.CONSCRIPT_ID
JOIN "HACKATHON_USR"."FI_DISEASE_INFO" di 
    ON ch.DISEASE_CODE = di.DISEASE_CODE
WHERE co.GARRISON = ?  -- Parameter: garrison name
  AND ch.RECORD_DATE >= ADD_DAYS(CURRENT_DATE, -90)
GROUP BY WEEK(ch.RECORD_DATE), YEAR(ch.RECORD_DATE), 
         ch.DISEASE_NAME, di.AVERAGE_DURATION_DAYS, di.PREVENTION
ORDER BY year DESC, week DESC, cases DESC
'''
        },
        
        "disease_information": {
            "description": "Get comprehensive disease information for agent recommendations",
            "sql": '''
SELECT 
    DISEASE_CODE,
    DISEASE_NAME_FI,
    DISEASE_NAME_EN,
    SYMPTOMS,
    PREVENTION,
    AVERAGE_DURATION_DAYS,
    HOW_IT_SPREADS,
    EPIDEMIC_THRESHOLD
FROM "HACKATHON_USR"."FI_DISEASE_INFO"
WHERE DISEASE_CODE = ?  -- Parameter: disease code
   OR DISEASE_NAME_FI = ?  -- Parameter: Finnish disease name
'''
        },
        
        "training_impact": {
            "description": "Calculate percentage of conscripts affected and training capacity impact",
            "sql": '''
SELECT 
    co.GARRISON,
    gi.CAPACITY,
    COUNT(DISTINCT co.CONSCRIPT_ID) as total_conscripts,
    COUNT(DISTINCT CASE 
        WHEN ch.RECORD_DATE >= ADD_DAYS(CURRENT_DATE, -7) 
        THEN ch.CONSCRIPT_ID 
    END) as currently_sick,
    ROUND(COUNT(DISTINCT CASE 
        WHEN ch.RECORD_DATE >= ADD_DAYS(CURRENT_DATE, -7) 
        THEN ch.CONSCRIPT_ID 
    END) * 100.0 / NULLIF(COUNT(DISTINCT co.CONSCRIPT_ID), 0), 2) as sick_percentage,
    CASE 
        WHEN ROUND(COUNT(DISTINCT CASE 
            WHEN ch.RECORD_DATE >= ADD_DAYS(CURRENT_DATE, -7) 
            THEN ch.CONSCRIPT_ID 
        END) * 100.0 / NULLIF(COUNT(DISTINCT co.CONSCRIPT_ID), 0), 2) > 20 
        THEN 'SEVERE IMPACT - Consider postponing training'
        WHEN ROUND(COUNT(DISTINCT CASE 
            WHEN ch.RECORD_DATE >= ADD_DAYS(CURRENT_DATE, -7) 
            THEN ch.CONSCRIPT_ID 
        END) * 100.0 / NULLIF(COUNT(DISTINCT co.CONSCRIPT_ID), 0), 2) > 10 
        THEN 'MODERATE IMPACT - Adjust training plans'
        ELSE 'MINIMAL IMPACT'
    END as training_recommendation
FROM "HACKATHON_USR"."FI_CONSCRIPT_ORGANIZATION" co
LEFT JOIN "HACKATHON_USR"."FI_CONSCRIPT_HEALTH" ch 
    ON co.CONSCRIPT_ID = ch.CONSCRIPT_ID
JOIN "HACKATHON_USR"."FI_GARRISON_INFO" gi 
    ON co.GARRISON = gi.GARRISON_NAME
GROUP BY co.GARRISON, gi.CAPACITY
'''
        },
        
        "regional_disease_comparison": {
            "description": "Compare disease patterns across regions and garrisons",
            "sql": '''
SELECT 
    thl.HYVINVOINTIALUE,
    thl.DISEASE_NAME,
    thl.YEAR,
    thl.WEEK,
    SUM(thl.CASE_COUNT) as regional_cases,
    COUNT(DISTINCT co.GARRISON) as garrisons_affected,
    STRING_AGG(DISTINCT co.GARRISON, ', ') as garrison_list
FROM "HACKATHON_USR"."FI_THL_DISEASE_CASES" thl
JOIN "HACKATHON_USR"."FI_MUNICIPALITY_MAPPING" mm 
    ON thl.HYVINVOINTIALUE = mm.HYVINVOINTIALUE
JOIN "HACKATHON_USR"."FI_CONSCRIPT_ORGANIZATION" co 
    ON mm.MUNICIPALITY = co.HOME_MUNICIPALITY
WHERE thl.YEAR = 2026 
  AND thl.WEEK >= WEEK(CURRENT_DATE) - 4
GROUP BY thl.HYVINVOINTIALUE, thl.DISEASE_NAME, thl.YEAR, thl.WEEK
ORDER BY regional_cases DESC
'''
        },
        
        "unit_outbreak_detection": {
            "description": "Detect potential outbreaks within specific units",
            "sql": '''
SELECT 
    co.GARRISON,
    co.UNIT,
    ch.DISEASE_NAME,
    COUNT(DISTINCT co.CONSCRIPT_ID) as unit_size,
    COUNT(DISTINCT ch.CONSCRIPT_ID) as infected_count,
    ROUND(COUNT(DISTINCT ch.CONSCRIPT_ID) * 100.0 / 
          NULLIF(COUNT(DISTINCT co.CONSCRIPT_ID), 0), 2) as infection_rate,
    di.PREVENTION,
    CASE 
        WHEN ROUND(COUNT(DISTINCT ch.CONSCRIPT_ID) * 100.0 / 
             NULLIF(COUNT(DISTINCT co.CONSCRIPT_ID), 0), 2) > 25 
        THEN 'QUARANTINE UNIT RECOMMENDED'
        WHEN ROUND(COUNT(DISTINCT ch.CONSCRIPT_ID) * 100.0 / 
             NULLIF(COUNT(DISTINCT co.CONSCRIPT_ID), 0), 2) > 15 
        THEN 'ENHANCED MONITORING REQUIRED'
        ELSE 'STANDARD PROTOCOLS'
    END as recommendation
FROM "HACKATHON_USR"."FI_CONSCRIPT_ORGANIZATION" co
LEFT JOIN "HACKATHON_USR"."FI_CONSCRIPT_HEALTH" ch 
    ON co.CONSCRIPT_ID = ch.CONSCRIPT_ID 
    AND ch.RECORD_DATE >= ADD_DAYS(CURRENT_DATE, -7)
LEFT JOIN "HACKATHON_USR"."FI_DISEASE_INFO" di 
    ON ch.DISEASE_CODE = di.DISEASE_CODE
GROUP BY co.GARRISON, co.UNIT, ch.DISEASE_NAME, di.PREVENTION
HAVING COUNT(DISTINCT ch.CONSCRIPT_ID) > 0
ORDER BY infection_rate DESC
'''
        }
    },
    
    "agent_question_mapping": {
        "Is there a possibility for an epidemic?": [
            "Query FI_CONSCRIPT_HEALTH for recent cases in last 14 days",
            "Compare count against FI_DISEASE_INFO.EPIDEMIC_THRESHOLD",
            "Check FI_THL_DISEASE_CASES for regional trends",
            "Use epidemic_detection query"
        ],
        
        "What kind of epidemic?": [
            "Join FI_CONSCRIPT_HEALTH with FI_DISEASE_INFO",
            "Return DISEASE_NAME_EN, SYMPTOMS, HOW_IT_SPREADS",
            "Use disease_information query"
        ],
        
        "What countermeasures could be taken?": [
            "Retrieve FI_DISEASE_INFO.PREVENTION for specific disease",
            "Calculate infected percentage using training_impact query",
            "If >25% in unit: suggest quarantine (unit_outbreak_detection)",
            "If >15% in garrison: suggest enhanced monitoring",
            "Provide specific prevention measures from disease info"
        ],
        
        "How could it affect training?": [
            "Use training_impact query",
            "Calculate sick_percentage per garrison/unit",
            "Use AVERAGE_DURATION_DAYS to project timeline",
            "Provide training_recommendation based on severity"
        ],
        
        "Are there disease trends in specific areas/garrisons?": [
            "Use garrison_disease_trends for garrison-level analysis",
            "Use regional_disease_comparison for multi-region view",
            "Analyze FI_THL_DISEASE_CASES time series by hyvinvointialue",
            "Cross-reference with HOME_MUNICIPALITY for weekend risk"
        ],
        
        "Does some disease trend in some garrisons?": [
            "Time-series analysis with garrison_disease_trends query",
            "Compare with national trends from FI_THL_DISEASE_CASES",
            "Identify garrison-specific patterns",
            "Use unit_outbreak_detection for unit-level granularity"
        ],
        
        "What is the current health status of a garrison?": [
            "Combine epidemic_detection and training_impact queries",
            "Show currently_sick count and sick_percentage",
            "List diseases present with unit_outbreak_detection",
            "Provide garrison capacity context from FI_GARRISON_INFO"
        ],
        
        "Should conscripts be prevented from weekend visits?": [
            "Use weekend_travel_risk query",
            "Check THL case counts in conscript home regions",
            "If home region CASE_COUNT > threshold: recommend restrictions",
            "Prioritize high-risk municipalities"
        ]
    },
    
    "key_metrics": {
        "epidemic_threshold_ranges": {
            "Influenssa": 30,
            "Rinovirus": 50,
            "Koronavirus": 20,
            "Norovirus": 15,
            "Adenovirus": 25
        },
        "training_impact_levels": {
            "minimal": "0-10% sick",
            "moderate": "10-20% sick - adjust training",
            "severe": ">20% sick - postpone training"
        },
        "quarantine_thresholds": {
            "unit_level": ">25% infection rate",
            "enhanced_monitoring": ">15% infection rate"
        }
    },
    
    "data_quality_notes": {
        "FI_CONSCRIPT_HEALTH": "Health records have random dates between 2025-01-01 and 2026-01-28 for simulation",
        "FI_THL_DISEASE_CASES": "THL data parsed from wide format; week numbers may need refinement for production",
        "FI_MUNICIPALITY_MAPPING": "Complete mapping of all Finnish municipalities to wellbeing counties"
    }
}

# Example usage in RAG agent:
# schema_context = FINLAND_HEALTH_DATA_MODEL
# agent_prompt = f"Schema: {schema_context['schema']}\nTables: {schema_context['tables'].keys()}\n..."